# AEGES-Q: QML Training and Evaluation

This notebook implements the actual quantum machine learning experiment
for the AEGES-Q cybersecurity framework.

The purpose is to train and evaluate a quantum machine learning classifier
using the UNSW-NB15 dataset.

The quantum model will be compared against the existing classical
Random Forest baseline using the same dataset split.

The comparison will consider:

1. Accuracy
2. Precision
3. Recall
4. F1-score
5. ROC-AUC
6. False-positive rate
7. Training time
8. Inference time
9. Circuit depth
10. Number of qubits

The final objective is to determine whether the quantum model is suitable
for integration into the adaptive AEGES-Q framework.

## 1. Environment Verification

This section verifies that the notebook is using the correct project
virtual environment and that all required quantum machine learning
libraries are available.

The experiments will initially run locally using the Qiskit Aer simulator.
No physical quantum computer is required for this stage.

In [1]:
import sys
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import qiskit
import qiskit_aer

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.circuit import ParameterVector

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

print("\nQiskit version:")
print(qiskit.__version__)

print("\nQiskit Aer version:")
print(qiskit_aer.__version__)

print("\nQiskit Machine Learning version:")
print(version("qiskit-machine-learning"))

simulator = AerSimulator()

print("\nAer simulator initialized successfully.")

Python executable:
c:\Projects\Aeges-Q\.venv\Scripts\python.exe

Python version:
3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]

Qiskit version:
2.5.2

Qiskit Aer version:
0.17.2

Qiskit Machine Learning version:
0.9.1

Aer simulator initialized successfully.


## 2. Locate the Existing UNSW-NB15 Dataset

The classical machine learning pipeline has already been completed.

This notebook will reuse the existing dataset and preprocessing outputs
instead of creating a separate preprocessing pipeline.

This is important because the quantum and classical models must be evaluated
on the same data split for a fair comparison.

The first step is to identify the existing dataset files inside the project.

In [4]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]

# Only show likely data/model files, excluding notebooks and cache folders
allowed_extensions = {
    ".csv", ".parquet", ".pkl", ".joblib", ".json", ".xlsx", ".npy", ".npz"
}

excluded_parts = {
    ".git",
    ".venv",
    "__pycache__",
    ".ipynb_checkpoints",
}

print("Project root:")
print(PROJECT_ROOT)

print("\nPossible dataset/model files:")

matches = []

for path in PROJECT_ROOT.rglob("*"):
    if not path.is_file():
        continue

    if path.suffix.lower() not in allowed_extensions:
        continue

    if any(part in excluded_parts for part in path.parts):
        continue

    matches.append(path)

for path in sorted(matches):
    print(path.relative_to(PROJECT_ROOT))

print(f"\nTotal matching files: {len(matches)}")

Project root:
c:\Projects\Aeges-Q

Possible dataset/model files:
artifacts\classical\random_forest.joblib
artifacts\classical\variant_e_preprocessor.joblib
data\raw\NUSW-NB15_features.csv
data\raw\UNSW-NB15_1.csv
data\raw\UNSW-NB15_2.csv
data\raw\UNSW-NB15_3.csv
data\raw\UNSW-NB15_4.csv
data\raw\UNSW-NB15_LIST_EVENTS.csv
data\raw\UNSW_NB15_testing-set.csv
data\raw\UNSW_NB15_training-set.csv

Total matching files: 10


## 3. Load Existing Dataset and Classical Artifacts

The classical IDS pipeline has already been trained and evaluated.

This notebook reuses the existing UNSW-NB15 training and testing datasets, preprocessing artifact, and Random Forest model so that the quantum model can be evaluated against the same baseline.

In [8]:
from pathlib import Path
import joblib
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parents[1]

TRAIN_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "UNSW_NB15_training-set.csv"
)

TEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "UNSW_NB15_testing-set.csv"
)

PREPROCESSOR_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "classical"
    / "variant_e_preprocessor.joblib"
)

RF_MODEL_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "classical"
    / "random_forest.joblib"
)

print("Training dataset:", TRAIN_PATH.exists())
print("Testing dataset:", TEST_PATH.exists())
print("Preprocessor:", PREPROCESSOR_PATH.exists())
print("Random Forest:", RF_MODEL_PATH.exists())

Training dataset: True
Testing dataset: True
Preprocessor: True
Random Forest: True


In [9]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

preprocessor = joblib.load(PREPROCESSOR_PATH)
random_forest = joblib.load(RF_MODEL_PATH)

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)
print("Preprocessor:", type(preprocessor).__name__)
print("Random Forest:", type(random_forest).__name__)

Training shape: (82332, 45)
Testing shape: (175341, 45)
Preprocessor: ColumnTransformer
Random Forest: RandomForestClassifier
